In [1]:
from __future__ import print_function
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms

In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # in_channels=1 --> black and white,  out_channels=32  --> filters,  kernel_size=3  --> (3, 3),  stride=1  --> jump from a pixel to another pixel
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [4]:
def train(model, device, data_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(data_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 10 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(data_loader.dataset)} ({100 * batch_idx / len(data_loader):.0f}%)]\tLoss: {loss.item():.6f}')


In [5]:
def test(model, device, data_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
        
    test_loss /+ len(data_loader.dataset)

    print(f'\nTest set: Avatage loss: {test_loss:.4f}, Accuracy: {correct}/{len(data_loader.dataset)} ({100 * correct / len(data_loader.dataset):.0f}%)\n')


In [ ]:
torch.manual_seed(42)
use_cuda = torch.cuda.is_available()

if use_cuda:
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

train_kwargs = {'batch_size': 64}
test_kwargs = {'batch_size': 1000}

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307), std=(0.3081))
])

dataset1 = datasets.MNIST('data', train=True, download=True, transform=transform)
dataset2 = datasets.MNIST('data', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(dataset=dataset1, **train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=1)

scheduler = StepLR(optimizer=optimizer, step_size=1, gamma=0.7)
for epoch in range(1, 2):
    train(model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), 'data/mnist_cnn.pt')

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.319619
Train Epoch: 1 [640/60000 (1%)]	Loss: 1.503810
Train Epoch: 1 [1280/60000 (2%)]	Loss: 0.775641
Train Epoch: 1 [1920/60000 (3%)]	Loss: 0.499190
Train Epoch: 1 [2560/60000 (4%)]	Loss: 0.274318
Train Epoch: 1 [3200/60000 (5%)]	Loss: 0.327446
Train Epoch: 1 [3840/60000 (6%)]	Loss: 0.262752
Train Epoch: 1 [4480/60000 (7%)]	Loss: 0.310988
Train Epoch: 1 [5120/60000 (9%)]	Loss: 0.615225
Train Epoch: 1 [5760/60000 (10%)]	Loss: 0.300353
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.191523
Train Epoch: 1 [7040/60000 (12%)]	Loss: 0.192827
Train Epoch: 1 [7680/60000 (13%)]	Loss: 0.207853
Train Epoch: 1 [8320/60000 (14%)]	Loss: 0.169854
Train Epoch: 1 [8960/60000 (15%)]	Loss: 0.204894
Train Epoch: 1 [9600/60000 (16%)]	Loss: 0.169487
Train Epoch: 1 [10240/60000 (17%)]	Loss: 0.324663
Train Epoch: 1 [10880/60000 (18%)]	Loss: 0.245657
Train Epoch: 1 [11520/60000 (19%)]	Loss: 0.379137
Train Epoch: 1 [12160/60000 (20%)]	Loss: 0.195878
Train Epoch: 1 [12800/60000 (